## Lab-3: Effect of L1, L2, and Elastic Net Regularization on Model Training
### Reg.No: 2548514

1) Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


2) Load Mushroom Dataset

In [2]:
df = pd.read_csv("mushrooms - mushrooms.csv")
df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


3) Basic Dataset Check

In [3]:
print("Shape:", df.shape)
print("\nTarget Distribution:\n", df["class"].value_counts())
print("\nMissing Values:\n", df.isnull().sum().sum())


Shape: (8124, 23)

Target Distribution:
 class
e    4208
p    3916
Name: count, dtype: int64

Missing Values:
 0


4) Split Features (X) and Target (y)

In [4]:
X = df.drop("class", axis=1)
y = df["class"]   # e or p


5) Train-Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (6499, 22)
Test shape: (1625, 22)


6) OneHot Encoding for Categorical Features

In [6]:
categorical_features = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)


7) Model 1 — Logistic Regression with L2 Regularization (Ridge)

In [7]:
l2_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000))
])

l2_model.fit(X_train, y_train)
y_pred_l2 = l2_model.predict(X_test)

print("L2 Regularization Test Accuracy:", accuracy_score(y_test, y_pred_l2))
print("\nClassification Report:\n", classification_report(y_test, y_pred_l2))


L2 Regularization Test Accuracy: 0.9993846153846154

Classification Report:
               precision    recall  f1-score   support

           e       1.00      1.00      1.00       842
           p       1.00      1.00      1.00       783

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625



8) Model 2 — Logistic Regression with L1 Regularization (Lasso)

In [8]:
l1_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(penalty="l1", solver="liblinear", max_iter=1000))
])

l1_model.fit(X_train, y_train)
y_pred_l1 = l1_model.predict(X_test)

print("L1 Regularization Test Accuracy:", accuracy_score(y_test, y_pred_l1))
print("\nClassification Report:\n", classification_report(y_test, y_pred_l1))


L1 Regularization Test Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           e       1.00      1.00      1.00       842
           p       1.00      1.00      1.00       783

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625



9) Model 3 — Logistic Regression with Elastic Net Regularization

In [9]:
elastic_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.5,     # 0.5 means 50% L1 + 50% L2
        max_iter=2000
    ))
])

elastic_model.fit(X_train, y_train)
y_pred_elastic = elastic_model.predict(X_test)

print("Elastic Net Test Accuracy:", accuracy_score(y_test, y_pred_elastic))
print("\nClassification Report:\n", classification_report(y_test, y_pred_elastic))


Elastic Net Test Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           e       1.00      1.00      1.00       842
           p       1.00      1.00      1.00       783

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625



10) Compare L1 vs L2 vs ElasticNet (Accuracy Table)

In [10]:
results = pd.DataFrame({
    "Regularization": ["L2 (Ridge)", "L1 (Lasso)", "Elastic Net"],
    "Test Accuracy": [
        accuracy_score(y_test, y_pred_l2),
        accuracy_score(y_test, y_pred_l1),
        accuracy_score(y_test, y_pred_elastic)
    ]
})

results


,Regularization,Test Accuracy
0,L2 (Ridge),0.999385
1,L1 (Lasso),1.000000
2,Elastic Net,1.000000


11) Confusion Matrix for All 3 Models

In [11]:
print("Confusion Matrix - L2:\n", confusion_matrix(y_test, y_pred_l2))
print("\nConfusion Matrix - L1:\n", confusion_matrix(y_test, y_pred_l1))
print("\nConfusion Matrix - ElasticNet:\n", confusion_matrix(y_test, y_pred_elastic))


Confusion Matrix - L2:
 [[842   0]
 [  1 782]]

Confusion Matrix - L1:
 [[842   0]
 [  0 783]]

Confusion Matrix - ElasticNet:
 [[842   0]
 [  0 783]]


Effect of Regularization Strength (C value)

In [12]:
C_values = [0.01, 0.1, 1, 10]

for c in C_values:
    model = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(penalty="l2", solver="lbfgs", C=c, max_iter=1000))
    ])
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"C={c}  -> Test Accuracy={accuracy_score(y_test, y_pred):.4f}")


C=0.01  -> Test Accuracy=0.9828
C=0.1  -> Test Accuracy=0.9969
C=1  -> Test Accuracy=0.9994
C=10  -> Test Accuracy=1.0000


### Interpretation:

L2 (Ridge)

Penalizes large weights

Keeps all features but reduces weight size

Good when many features contribute slightly

L1 (Lasso)

Penalizes absolute weights

Pushes many weights to exactly 0

Performs feature selection

Can reduce overfitting and simplify model

Elastic Net

Mix of L1 + L2

Useful when dataset has many correlated features

More stable than pure L1